# HumAID — Dataset Explorer

## Key settings
- `MODEL` (e.g., `gpt-4o`), `RULES` (e.g., `RULES_1`), `TAG`
- `DRYRUN_N`, `POLL_SECS`
- Token budgeting: `BATCH_TOKEN_LIMIT`, `SAFETY_MARGIN`, `MAX_OUTPUT_TOKENS`
- `.env` with `OPENAI_API_KEY_1` (and optionally a second key)

# 0) Setup

In [1]:
from pathlib import Path
import math
import pandas as pd
from dotenv import load_dotenv; load_dotenv()

from humaidclf import run_experiment
from humaidclf import build_token_index               # token budgeting (sampling-based)
from humaidclf import run_experiment_sharded          # NEW: stratified sharded runner
from humaidclf.batch import use_api_key_env           # (optional) keep key switcher
from rules import RULES_1

# --- config ---
BASE = Path("Dataset/HumAID")
SPLITS = ["train"]             # or ["train","dev","test"]
MODEL = "gpt-4o"
RULES = RULES_1
TAG = "modeS-gpt-4o-RULES1-filtered"
DRYRUN_N = 20
POLL_SECS = 300
DO_ANALYSIS = True
OUT_ROOT = "runs"

# Token caps & estimates
BATCH_TOKEN_LIMIT = 90_000   # Tier-1 cap
SAFETY_MARGIN = 0.90            # use only 90% of the cap
MAX_OUTPUT_TOKENS = 40          # matches your request schema

# Discover datasets (events/splits)

In [2]:
def discover_tsvs(base: Path, splits: list[str]):
    items = []
    for event_dir in sorted([p for p in base.iterdir() if p.is_dir()]):
        event = event_dir.name
        for split in splits:
            tsv = event_dir / f"{event}_{split}.tsv"
            if tsv.exists():
                items.append({"event": event, "split": split, "tsv": str(tsv)})
    return pd.DataFrame(items)

df_sources = discover_tsvs(BASE, SPLITS)

# --- token budgeting ---
token_index = build_token_index(
    df_sources,
    model=MODEL,
    rules_text=RULES,
    batch_token_limit=BATCH_TOKEN_LIMIT,
    safety_margin=SAFETY_MARGIN,
    sample_size=200,
    max_output_tokens=MAX_OUTPUT_TOKENS,
)

display(token_index)

df_fit     = token_index[token_index["fits_cap"]].reset_index(drop=True)
df_too_big = token_index[~token_index["fits_cap"]].reset_index(drop=True)

print("OK to run as single batch:")
display(df_fit[["event","split","num_rows","est_total_tokens","limit_used_%"]])

print("Will be sharded (exceeds cap):")
display(df_too_big[["event","split","num_rows","est_total_tokens","limit_used_%"]])



print("\n=== Per-dataset label distributions (class_label) ===")

for _, src in df_sources.iterrows():
    tsv_path = Path(src["tsv"])
    event    = src["event"]
    split    = src["split"]

    df = pd.read_csv(tsv_path, sep="\t")

    if "class_label" not in df.columns:
        print(f"\n--- {event} / {split} ---")
        print("No 'class_label' column in this TSV; skipping.")
        continue

    # Clean up labels a bit (match eval logic: ignore blanks / nan / None)
    labels = (
        df["class_label"]
        .astype(str)
        .str.strip()
        .replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
        .dropna()
    )

    counts = (
        labels.value_counts()
        .rename_axis("class_label")
        .reset_index(name="count")
    )

    print(f"\n--- {event} / {split} ---")
    print(f"TSV: {tsv_path}")
    print(f"Total rows: {len(df)}  |  Labeled rows: {len(labels)}")
    display(counts)


,event,split,tsv,num_rows,avg_req_tokens,est_total_tokens,fits_cap,limit_used_%
1,canada_wildfires_2016,train,Dataset\HumAID\canada_wildfires_2016\canada_wi...,1569,473,742137,False,824.6
8,kaikoura_earthquake_2016,train,Dataset\HumAID\kaikoura_earthquake_2016\kaikou...,1536,486,746496,False,829.4
2,cyclone_idai_2019,train,Dataset\HumAID\cyclone_idai_2019\cyclone_idai_...,2753,518,1426054,False,1584.5
4,hurricane_florence_2018,train,Dataset\HumAID\hurricane_florence_2018\hurrica...,4384,501,2196384,False,2440.4
7,hurricane_maria_2017,train,Dataset\HumAID\hurricane_maria_2017\hurricane_...,5094,488,2485872,False,2762.1
0,california_wildfires_2018,train,Dataset\HumAID\california_wildfires_2018\calif...,5163,508,2622804,False,2914.2
3,hurricane_dorian_2019,train,Dataset\HumAID\hurricane_dorian_2019\hurricane...,5329,501,2669829,False,2966.5
9,kerala_floods_2018,train,Dataset\HumAID\kerala_floods_2018\kerala_flood...,5588,506,2827528,False,3141.7
5,hurricane_harvey_2017,train,Dataset\HumAID\hurricane_harvey_2017\hurricane...,6378,486,3099708,False,3444.1
6,hurricane_irma_2017,train,Dataset\HumAID\hurricane_irma_2017\hurricane_i...,6579,486,3197394,False,3552.7


OK to run as single batch:


,event,split,num_rows,est_total_tokens,limit_used_%


Will be sharded (exceeds cap):


,event,split,num_rows,est_total_tokens,limit_used_%
0,canada_wildfires_2016,train,1569,742137,824.6
1,kaikoura_earthquake_2016,train,1536,746496,829.4
2,cyclone_idai_2019,train,2753,1426054,1584.5
3,hurricane_florence_2018,train,4384,2196384,2440.4
4,hurricane_maria_2017,train,5094,2485872,2762.1
5,california_wildfires_2018,train,5163,2622804,2914.2
6,hurricane_dorian_2019,train,5329,2669829,2966.5
7,kerala_floods_2018,train,5588,2827528,3141.7
8,hurricane_harvey_2017,train,6378,3099708,3444.1
9,hurricane_irma_2017,train,6579,3197394,3552.7


...

,class_label,count
0,rescue_volunteering_or_donation_effort,1384
1,other_relevant_information,1097
2,infrastructure_and_utility_damage,999
3,requests_or_urgent_needs,498
4,sympathy_and_support,470
5,injured_or_dead_people,211
6,not_humanitarian,189
7,caution_and_advice,154
8,displaced_people_and_evacuations,92



--- kaikoura_earthquake_2016 / train ---
TSV: Dataset\HumAID\kaikoura_earthquake_2016\kaikoura_earthquake_2016_train.tsv
Total rows: 1536  |  Labeled rows: 1536


,class_label,count
0,caution_and_advice,345
1,sympathy_and_support,302
2,other_relevant_information,218
3,infrastructure_and_utility_damage,218
4,not_humanitarian,157
5,rescue_volunteering_or_donation_effort,145
6,injured_or_dead_people,73
7,displaced_people_and_evacuations,61
8,requests_or_urgent_needs,17



--- kerala_floods_2018 / train ---
TSV: Dataset\HumAID\kerala_floods_2018\kerala_floods_2018_train.tsv
Total rows: 5588  |  Labeled rows: 5588


,class_label,count
0,rescue_volunteering_or_donation_effort,3005
1,other_relevant_information,669
2,sympathy_and_support,585
3,requests_or_urgent_needs,413
4,not_humanitarian,319
5,injured_or_dead_people,254
6,infrastructure_and_utility_damage,207
7,caution_and_advice,97
8,displaced_people_and_evacuations,39
